# Energy usage prediction: ensemble model (`candidate_ensemble.csv`)

This notebook builds our latest submission and explains every step. It blends **four models** that fill in missing values in different ways, so they make different mistakes and partly cancel each other out.

```
  v4  (Part B) ─────────────────────────────┐ 30%
                                            ├──► "our model" ─────────┐ 75%
  LightGBM-imputed model (Part C, 10 seeds) ┐│ 70%                    │
                                            ├┘ (50/50)                ├──► candidate_ensemble.csv
  CatBoost-imputed model (Part D) ──────────┘                         │
                                                                      │ 25%
  Friend's model (Part E: CatBoost imputers + Ridge + CatBoost) ──────┘
```

**How the weights were chosen:** all of them came from cross-validation on `train.csv`, with test-style blanks and the test set's building/event mix. None came from leaderboard scores. The two imputer versions are averaged 50/50 with no tuning.

**Rules we followed:** test answers are never used. Test inputs are used without labels: to train the imputers and to weight rows toward the test mix.

**Running it:** the models need LightGBM (Parts B–C) and CatBoost (Parts D–E). Neither of our environments has both, so each model's predictions are saved under `experiments/`. With `RETRAIN_COMPONENTS = False` (the default), the notebook loads them and reproduces `candidate_ensemble.csv` exactly. Set it to `True` to retrain every model that the current environment's libraries allow.

## Part A: settings, data and shared helpers

In [1]:
import importlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.experimental import enable_iterative_imputer  # must be imported before IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

warnings.filterwarnings('ignore')


def optional_import(name):
    try:
        return importlib.import_module(name)
    except ImportError:
        return None


lightgbm = optional_import('lightgbm')   # needed only to retrain Part C
catboost = optional_import('catboost')   # needed only to retrain Parts D and E

# ---------------- settings you might want to change ----------------
DATA_FOLDER = Path('.')                          # folder that contains train.csv and test.csv
SAVED_FOLDER = DATA_FOLDER / 'experiments'       # saved predictions of each model
OUTPUT_FILE = DATA_FOLDER / 'candidate_ensemble.csv'
RETRAIN_V4 = False            # False: reuse candidate_v4.csv (exact v4 predictions that scored 3.37)
RETRAIN_COMPONENTS = False    # False: load saved predictions; True: retrain the models this environment can run
RANDOM_SEEDS = list(range(527, 537))             # 10 seeds for the LightGBM-imputed model
CPU_THREADS = 4

# blend weights (all chosen by cross-validation on train.csv, not on the leaderboard)
WEIGHT_V4_IN_OUR_MODEL = 0.30        # our model = 30% v4 + 70% imputed models
WEIGHT_FRIEND_MODEL = 0.25           # final = 75% our model + 25% friend's model

print('scikit-learn', sklearn.__version__, '| pandas', pd.__version__,
      '| lightgbm', getattr(lightgbm, '__version__', 'not installed'),
      '| catboost', getattr(catboost, '__version__', 'not installed'))

scikit-learn 1.5.2 | pandas 2.2.3 | lightgbm 4.5.0 | catboost not installed


In [2]:
train = pd.read_csv(DATA_FOLDER / 'train.csv')
test = pd.read_csv(DATA_FOLDER / 'test.csv')
test_inputs = test.drop(columns=['energy_usage'], errors='ignore')   # test.csv has no answers anyway

NUMERIC_COLUMNS = ['temperature', 'humidity', 'occupancy', 'previous_usage']   # the only columns with blanks
BUILDING_IDS = ['ADM_A', 'BUS_A', 'BUS_B', 'ENG_A', 'ENG_B', 'LEC_A', 'LIB_A', 'RES_A', 'RES_B', 'SCI_A', 'SCI_B', 'SPT_A']
BUILDING_TYPES = ['Administration', 'Business', 'Engineering', 'LectureHall', 'Library', 'Residential', 'Science', 'Sports']
DAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
DAY_NUMBER = {day: number for number, day in enumerate(DAY_NAMES)}
BUILDING_NUMBER = {building: number for number, building in enumerate(BUILDING_IDS)}

print(f'train: {len(train):,} rows | test: {len(test):,} rows')
print('blank values in test:', test_inputs[NUMERIC_COLUMNS].isna().sum().to_dict())

train: 8,000 rows | test: 3,000 rows
blank values in test: {'temperature': 211, 'humidity': 183, 'occupancy': 206, 'previous_usage': 203}


### Unusual rows and test-mix weights

The test set has about **five times as many unusual "event" rows** as the training set. Each row is compared with the typical value for its building, hour and weekday/weekend, and the first matching event is recorded:

| Event number | Event | Rule |
|---|---|---|
| 1 | heat | temperature of 33.5 or more |
| 2 | rain | humidity of 93 or more |
| 3 | occupancy surge | 100 or more people above typical |
| 4 | occupancy drop | 60 or more people below typical |
| 5 | previous-usage spike | more than 1.6 times typical |
| 6 | previous-usage drop | less than half of typical |

`test_mix_weights` gives each training row a weight, so the weighted training data has the same building × event mix as the test inputs. The counts are smoothed toward the overall event mix, so rare combinations don't get extreme weights, and every weight is clipped to between 0.2 and 10.

In [3]:
def is_weekend(rows):
    return rows['day_of_week'].isin(['Saturday', 'Sunday'])


def event_type(rows, reference_rows):
    """0 = normal row, 1-6 = the first event (see table above) that applies to the row."""
    group_keys = ['building_id', 'hour', 'weekend']
    reference = reference_rows.assign(weekend=is_weekend(reference_rows))
    typical = reference.groupby(group_keys)[['occupancy', 'previous_usage']].median().add_suffix('_typical')
    joined = rows.assign(weekend=is_weekend(rows)).join(typical, on=group_keys)

    occupancy_gap = joined['occupancy'] - joined['occupancy_typical']
    previous_ratio = joined['previous_usage'] / joined['previous_usage_typical']
    event_flags = np.column_stack([
        joined['temperature'] >= 33.5,     # 1 heat
        joined['humidity'] >= 93,          # 2 rain
        occupancy_gap >= 100,              # 3 occupancy surge
        occupancy_gap <= -60,              # 4 occupancy drop
        previous_ratio > 1.6,              # 5 previous-usage spike
        previous_ratio < 0.5,              # 6 previous-usage drop
    ])                                     # blank values count as "no event"
    first_event = np.argmax(event_flags, axis=1) + 1
    return pd.Series(np.where(event_flags.any(axis=1), first_event, 0), index=rows.index)


def test_mix_weights(source_rows, target_rows, smoothing=15):
    """Weight for each source row so the source's building x event mix matches the target's."""
    source_events = event_type(source_rows, source_rows)
    target_events = event_type(target_rows, source_rows)       # "typical" values always come from the source rows
    all_events = range(7)

    source_counts = pd.crosstab(source_rows['building_id'], source_events).reindex(index=BUILDING_IDS, columns=all_events, fill_value=0)
    target_counts = pd.crosstab(target_rows['building_id'], target_events).reindex(index=BUILDING_IDS, columns=all_events, fill_value=0)

    # share of each event within each building, smoothed toward the overall event mix
    source_overall_mix = (np.bincount(source_events, minlength=7) + 1) / (len(source_events) + 7)
    target_overall_mix = (np.bincount(target_events, minlength=7) + 1) / (len(target_events) + 7)
    source_event_share = (source_counts + smoothing * source_overall_mix).div(source_counts.sum(axis=1) + smoothing, axis=0)
    target_event_share = (target_counts + smoothing * target_overall_mix).div(target_counts.sum(axis=1) + smoothing, axis=0)
    event_ratio = target_event_share / source_event_share

    # how much more (or less) common each building is in the target
    building_ratio = (target_counts.sum(axis=1) / len(target_rows)) / (source_counts.sum(axis=1) / len(source_rows))

    weights = [event_ratio.loc[building, event] * building_ratio[building]
               for building, event in zip(source_rows['building_id'], source_events)]
    return np.clip(np.array(weights), 0.2, 10.0)


train_events = event_type(train, train)
test_events = event_type(test_inputs, train)
print(f'unusual event rows: train {(train_events > 0).mean():.1%} | test {(test_events > 0).mean():.1%}')

unusual event rows: train 5.1% | test 26.1%


## Part B: the "v4" model (scored 3.37 on its own)

v4 blends two models: **25% the v3 notebook model** and **75% one model per pattern of missing inputs**.

### B1. The v3 notebook model

1. **Fill blanks with MissForest.** An `IterativeImputer` with extra-trees models predicts each blank value from the row's other values. It learns from training rows only.
2. **Ridge regression** on hand-made features:
   * each building gets its own slope for occupancy, previous usage, temperature and time of day;
   * weekend effects per building type;
   * how unusual the row is compared with its building's typical values (event features);
   * occupancy effects by part of the day, per building type.
3. **Gradient boosting** predicts the *change* from `previous_usage`, then adds `previous_usage` back.
4. **Blend:** 80% Ridge + 20% gradient boosting.

In [4]:
def missforest_inputs(rows):
    """The four numeric columns (may contain blanks) plus never-blank time and building columns."""
    inputs = rows[NUMERIC_COLUMNS].copy()
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    inputs['is_weekend'] = is_weekend(rows).astype(float)
    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    building_onehot.index = rows.index
    return pd.concat([inputs, building_onehot], axis=1)


def fit_missforest(training_rows):
    imputer = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=100, min_samples_leaf=3, max_features=0.6, n_jobs=CPU_THREADS, random_state=0),
        max_iter=8, random_state=0)
    return imputer.fit(missforest_inputs(training_rows))


def fill_blanks_with_missforest(imputer, rows):
    filled = rows.copy()
    filled[NUMERIC_COLUMNS] = imputer.transform(missforest_inputs(rows))[:, :len(NUMERIC_COLUMNS)]
    return filled


def typical_values(filled_rows):
    """Median occupancy and previous usage for each building, hour and weekday/weekend."""
    return (filled_rows.assign(is_weekend=is_weekend(filled_rows))
            .groupby(['building_id', 'hour', 'is_weekend'])[['occupancy', 'previous_usage']].median())


def ridge_features(rows, typical):
    """Features for the Ridge model (rows must have no blanks)."""
    features = rows[NUMERIC_COLUMNS].copy()
    features['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    features['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    features['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    features['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    features['is_weekend'] = is_weekend(rows).astype(float)
    features['occupancy_x_weekend'] = features['occupancy'] * features['is_weekend']

    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    type_onehot = pd.get_dummies(pd.Categorical(rows['building_type'], categories=BUILDING_TYPES), prefix='bt').astype(float)
    building_onehot.index = type_onehot.index = rows.index

    interactions = {}
    # each building gets its own slope for the main drivers
    for column in ['occupancy', 'previous_usage', 'temperature', 'hour_sin', 'hour_cos']:
        for building in building_onehot.columns:
            interactions[f'{column}_x_{building}'] = features[column] * building_onehot[building]
    # weekend effect per building type
    for column in ['is_weekend', 'occupancy_x_weekend']:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = features[column] * type_onehot[building_type]

    # how unusual the row is compared with its building's typical values
    lookup_keys = pd.DataFrame({'building_id': rows['building_id'], 'hour': rows['hour'], 'is_weekend': features['is_weekend'] > 0})
    typical_for_row = lookup_keys.join(typical, on=['building_id', 'hour', 'is_weekend']).fillna(typical.median())
    events = pd.DataFrame({
        'occupancy_gap': features['occupancy'] - typical_for_row['occupancy'],
        'previous_ratio': features['previous_usage'] / typical_for_row['previous_usage'],
        'previous_gap': features['previous_usage'] - typical_for_row['previous_usage'],
        'heat': (features['temperature'] >= 33.5).astype(float),
        'rain': (features['humidity'] >= 93).astype(float),
    }, index=rows.index)
    for column in events.columns:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = events[column] * type_onehot[building_type]

    # occupancy effect by part of the day, per building type
    part_of_day = pd.cut(rows['hour'], [-1, 5, 8, 11, 14, 17, 20, 23], labels=False)
    part_onehot = pd.get_dummies(pd.Categorical(part_of_day, categories=range(7)), prefix='dp').astype(float)
    part_onehot.index = rows.index
    for part in part_onehot.columns:
        for building_type in type_onehot.columns:
            interactions[f'occupancy_{part}_x_{building_type}'] = features['occupancy'] * part_onehot[part] * type_onehot[building_type]

    return pd.concat([features, type_onehot, building_onehot, events, pd.DataFrame(interactions, index=rows.index)], axis=1)


def tree_features(rows):
    """Features for the gradient-boosting model (raw values; building columns treated as categories)."""
    features = rows[['hour', 'month'] + NUMERIC_COLUMNS].copy()
    features['day'] = rows['day_of_week'].map(DAY_NUMBER)
    features['is_weekend'] = (features['day'] >= 5).astype(int)
    features['building_type'] = pd.Categorical(rows['building_type'], categories=BUILDING_TYPES)
    features['building_id'] = pd.Categorical(rows['building_id'], categories=BUILDING_IDS)
    return features


def new_gradient_boosting():
    return HistGradientBoostingRegressor(categorical_features='from_dtype', max_iter=1500, learning_rate=0.03,
                                         max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.0,
                                         early_stopping=True, n_iter_no_change=50, random_state=42)


class V3NotebookModel:
    def fit(self, training_rows):
        self.imputer = fit_missforest(training_rows)                    # learns from training rows only
        filled = fill_blanks_with_missforest(self.imputer, training_rows)
        target = filled['energy_usage']
        self.typical = typical_values(filled)
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(ridge_features(filled, self.typical), target)
        self.boosting = new_gradient_boosting()
        self.boosting.fit(tree_features(filled), target - filled['previous_usage'])     # predicts the change
        return self

    def predict(self, rows):
        filled = fill_blanks_with_missforest(self.imputer, rows)
        ridge_prediction = self.ridge.predict(ridge_features(filled, self.typical))
        boosting_prediction = self.boosting.predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
        return 0.8 * ridge_prediction + 0.2 * boosting_prediction

### B2. One model per pattern of missing inputs

Each of the four numeric columns can be present or blank, so there are **16 patterns**. For each pattern, we train a separate model that uses **only the columns available in that pattern**. A test row missing `previous_usage` is then predicted by a model that never relied on `previous_usage`, rather than from a guessed value. Research on prediction with missing data supports this "pattern submodel" approach.

Each pattern model is:

* a **Ridge regression** on the available numeric columns, a weekend flag, three daily sine/cosine curves and a yearly sine/cosine curve, plus each of these **per building**;
* plus a **gradient-boosting correction** (weight 0.65) trained on what the Ridge model got wrong.

Training rows are weighted toward the test mix (`test_mix_weights` raised to the power 0.5, a gentle version).

In [5]:
def pattern_model_inputs(rows, available_columns, n_daily_curves):
    """Available numeric columns, a weekend flag, daily sine/cosine curves and a yearly sine/cosine curve."""
    hour = rows['hour'].to_numpy(float)
    columns = {name: rows[name].to_numpy(float) for name in available_columns}
    columns['weekend'] = is_weekend(rows).to_numpy(float)
    for k in range(1, n_daily_curves + 1):                     # k = 1, 2, 3: once, twice, three times a day
        columns[f'hour_sin_{k}'] = np.sin(2 * np.pi * k * hour / 24)
        columns[f'hour_cos_{k}'] = np.cos(2 * np.pi * k * hour / 24)
    columns['month_sin'] = np.sin(2 * np.pi * rows['month'].to_numpy() / 12)
    columns['month_cos'] = np.cos(2 * np.pi * rows['month'].to_numpy() / 12)
    return np.column_stack(list(columns.values()))


def blank_pattern_number(rows):
    """0-15: bit k is set when NUMERIC_COLUMNS[k] is blank (0 = nothing blank)."""
    blanks = rows[NUMERIC_COLUMNS].isna().to_numpy().astype(int)
    return (blanks * 2 ** np.arange(len(NUMERIC_COLUMNS))).sum(axis=1)


class MissingPatternModels:
    def __init__(self, n_daily_curves=3, ridge_alpha=1.0, correction_weight=0.65,
                 correction_trees=250, correction_min_leaf=25, correction_max_leaves=15, test_mix_power=0.5):
        self.n_daily_curves = n_daily_curves
        self.ridge_alpha = ridge_alpha
        self.correction_weight = correction_weight
        self.correction_trees = correction_trees
        self.correction_min_leaf = correction_min_leaf
        self.correction_max_leaves = correction_max_leaves
        self.test_mix_power = test_mix_power

    def _ridge_inputs(self, rows, available_columns, scaler):
        scaled = scaler.transform(pattern_model_inputs(rows, available_columns, self.n_daily_curves))
        building_onehot = (rows['building_id'].to_numpy()[:, None] == np.array(BUILDING_IDS)[None, :]).astype(float)
        per_building = (scaled[:, :, None] * building_onehot[:, None, :]).reshape(len(scaled), -1)   # every input x every building
        return np.column_stack([scaled, building_onehot, per_building])

    @staticmethod
    def _correction_inputs(rows, available_columns):
        numeric = rows[['hour', 'month'] + available_columns].to_numpy(float)
        return np.column_stack([numeric, rows['day_of_week'].map(DAY_NUMBER), rows['building_id'].map(BUILDING_NUMBER)])

    def fit(self, training_rows, test_rows):
        weights = test_mix_weights(training_rows, test_rows) ** self.test_mix_power
        weights = pd.Series(weights / weights.mean(), index=training_rows.index)
        self.models = {}
        for pattern in range(2 ** len(NUMERIC_COLUMNS)):
            available = [column for bit, column in enumerate(NUMERIC_COLUMNS) if not pattern & (1 << bit)]
            usable_rows = training_rows.dropna(subset=available)          # rows that have every available column
            scaler = StandardScaler().fit(pattern_model_inputs(usable_rows, available, self.n_daily_curves))
            ridge_inputs = self._ridge_inputs(usable_rows, available, scaler)
            row_weights = weights.loc[usable_rows.index].to_numpy()

            ridge = Ridge(alpha=self.ridge_alpha).fit(ridge_inputs, usable_rows['energy_usage'], sample_weight=row_weights)
            correction_inputs = self._correction_inputs(usable_rows, available)
            correction = HistGradientBoostingRegressor(
                max_iter=self.correction_trees, max_leaf_nodes=self.correction_max_leaves, learning_rate=0.05,
                min_samples_leaf=self.correction_min_leaf, l2_regularization=8.0, early_stopping=False, random_state=42,
                categorical_features=[False] * (correction_inputs.shape[1] - 1) + [True])   # last column = building
            correction.fit(correction_inputs, usable_rows['energy_usage'].to_numpy() - ridge.predict(ridge_inputs),
                           sample_weight=row_weights)
            self.models[pattern] = (available, scaler, ridge, correction)
        return self

    def predict(self, rows):
        patterns = blank_pattern_number(rows)
        prediction = np.empty(len(rows))
        for pattern in np.unique(patterns):
            positions = np.flatnonzero(patterns == pattern)
            pattern_rows = rows.iloc[positions]
            available, scaler, ridge, correction = self.models[pattern]
            prediction[positions] = (ridge.predict(self._ridge_inputs(pattern_rows, available, scaler))
                                     + self.correction_weight * correction.predict(self._correction_inputs(pattern_rows, available)))
        return prediction

### B3. Put v4 together

**v4 = 25% v3 notebook model + 75% pattern models.**

`candidate_v4.csv` was produced with newer library versions than the platform's. Retrained here, v4 differs from that file by about 0.1 per row on average. To reproduce `candidate_final.csv` exactly, keep `RETRAIN_V4 = False`. Set it to `True` to build everything from `train.csv` and `test.csv` alone.

In [6]:
def v4_predictions(training_rows, test_rows):
    v3_model = V3NotebookModel().fit(training_rows)
    pattern_models = MissingPatternModels().fit(training_rows, test_rows)
    return 0.25 * v3_model.predict(test_rows) + 0.75 * pattern_models.predict(test_rows)


saved_v4_file = DATA_FOLDER / 'candidate_v4.csv'
if not RETRAIN_V4 and saved_v4_file.exists():
    saved_v4 = pd.read_csv(saved_v4_file)
    assert saved_v4['id'].equals(test['id']), 'candidate_v4.csv rows are not in test.csv order'
    v4_prediction = saved_v4['prediction'].to_numpy()
    print('v4: loaded candidate_v4.csv (the exact predictions that scored 3.37)')
else:
    with threadpool_limits(limits=CPU_THREADS):
        v4_prediction = v4_predictions(train, test_inputs)
    print('v4: retrained from scratch')

v4: loaded candidate_v4.csv (the exact predictions that scored 3.37)


## Part C: the LightGBM-imputed model (70% of the final prediction)

### C1. Four LightGBM models that fill in blank values

For each numeric column, a LightGBM model learns to predict it from the row's other values: building, building type, hour, day, month and the other numeric columns. Three details make these "imputers" suit the test set:

1. **They learn from the inputs of train and test rows together.** They predict input columns, never `energy_usage`, so no answers are involved.
2. **They practise on test-style blanks.** Every row is copied three more times, with blank patterns copied from real test rows. The column being learned always stays filled in. Test rows often have two or three blanks, which training rows almost never have.
3. **They focus on the test set's kind of rows**, using `test_mix_weights` raised to the power 1.5.

### C2. The energy model

After filling the blanks:

* **Ridge regression** on the same features as the v3 model, plus the imputers' **expected** `previous_usage` and `occupancy` for each row (each also per building type);
* **gradient boosting** on the change from `previous_usage`;
* **blend:** 80% Ridge + 20% gradient boosting.

In [7]:
IMPUTER_CATEGORIES = {'building_id': BUILDING_IDS, 'building_type': BUILDING_TYPES, 'day_of_week': DAY_NAMES}


def imputer_inputs(rows):
    """Everything the imputers may look at (the column being predicted is dropped later)."""
    inputs = rows[['building_id', 'building_type', 'hour', 'day_of_week', 'month'] + NUMERIC_COLUMNS].copy()
    inputs['weekend'] = is_weekend(rows).astype(int)
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    for column, categories in IMPUTER_CATEGORIES.items():          # text columns -> LightGBM categories
        inputs[column] = pd.Categorical(inputs[column], categories=categories)
    return inputs


def new_lightgbm_imputer(seed):
    return lightgbm.LGBMRegressor(n_estimators=2200, learning_rate=0.025, max_depth=5, num_leaves=32,
                                  min_child_samples=30, reg_lambda=15.0, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, cat_smooth=20, verbose=-1, random_state=seed, n_jobs=CPU_THREADS)


class LightGBMImputedModel:
    def __init__(self, seed, practice_copies=3, test_mix_power=1.5, boosting_share=0.2):
        self.seed = seed
        self.practice_copies = practice_copies
        self.test_mix_power = test_mix_power
        self.boosting_share = boosting_share

    # ---- C1: the imputers ----
    def fit_imputers(self, training_rows, test_rows):
        all_inputs = pd.concat([training_rows.drop(columns='energy_usage'), test_rows], ignore_index=True)
        random_generator = np.random.default_rng(self.seed)
        weights = test_mix_weights(all_inputs, test_rows) ** self.test_mix_power
        weights = weights / weights.mean()
        test_blank_patterns = test_rows[NUMERIC_COLUMNS].isna().to_numpy()

        self.imputers = {}
        for column in NUMERIC_COLUMNS:
            observed = all_inputs.loc[all_inputs[column].notna()].copy()   # rows where this column is known
            practice_sets = [observed]
            for _ in range(self.practice_copies):
                copy = observed.copy()
                blanks = test_blank_patterns[random_generator.integers(0, len(test_rows), len(copy))]
                blanks[:, NUMERIC_COLUMNS.index(column)] = False            # keep the column being learned
                copy[NUMERIC_COLUMNS] = copy[NUMERIC_COLUMNS].mask(blanks)
                practice_sets.append(copy)
            practice_data = pd.concat(practice_sets, ignore_index=True)
            practice_weights = np.tile(weights[observed.index.to_numpy()], self.practice_copies + 1)

            imputer = new_lightgbm_imputer(self.seed)
            imputer.fit(imputer_inputs(practice_data).drop(columns=column), practice_data[column], sample_weight=practice_weights)
            self.imputers[column] = imputer
        return self

    def expected_value(self, rows, column):
        """What the imputer thinks `column` should be, given the rest of the row."""
        return self.imputers[column].predict(imputer_inputs(rows).drop(columns=column))

    def fill_blanks(self, rows):
        filled = rows.copy()
        for column in NUMERIC_COLUMNS:
            blank = rows[column].isna()
            if blank.any():
                filled.loc[blank, column] = self.expected_value(rows.loc[blank], column)
        return filled

    # ---- C2: the energy model ----
    def model_features(self, filled_rows, original_rows):
        features = ridge_features(filled_rows, self.typical).copy()
        expected_previous = self.expected_value(original_rows, 'previous_usage')
        expected_occupancy = self.expected_value(original_rows, 'occupancy')
        features['expected_previous'] = expected_previous
        features['expected_occupancy'] = expected_occupancy
        for building_type in BUILDING_TYPES:
            this_type = (filled_rows['building_type'] == building_type).to_numpy(float)
            features[f'expected_previous_{building_type}'] = expected_previous * this_type
            features[f'expected_occupancy_{building_type}'] = expected_occupancy * this_type
        return features

    def fit(self, training_rows, test_rows):
        self.fit_imputers(training_rows, test_rows)
        filled = self.fill_blanks(training_rows)
        self.typical = typical_values(filled)
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(self.model_features(filled, training_rows), filled['energy_usage'])
        self.boosting = new_gradient_boosting()
        self.boosting.fit(tree_features(filled), filled['energy_usage'] - filled['previous_usage'])
        return self

    def predict(self, rows):
        filled = self.fill_blanks(rows)
        ridge_prediction = self.ridge.predict(self.model_features(filled, rows))
        boosting_prediction = self.boosting.predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
        return (1 - self.boosting_share) * ridge_prediction + self.boosting_share * boosting_prediction

### C3. Average 10 random seeds

The seed controls which test blank patterns go into the imputers' practice copies. On the public leaderboard, this model's score moved by about 0.02 between seeds. That's luck, so averaging 10 seeds removes it.

In [8]:
saved_lightgbm = SAVED_FOLDER / 'lightgbm_model_10_seeds.npy'
if RETRAIN_COMPONENTS and lightgbm is not None:
    seed_predictions = []
    with threadpool_limits(limits=CPU_THREADS):
        for seed in RANDOM_SEEDS:
            seed_predictions.append(LightGBMImputedModel(seed=seed).fit(train, test_inputs).predict(test_inputs))
            print(f'seed {seed} done')
    seed_predictions = np.vstack(seed_predictions)
    print('LightGBM-imputed model: retrained')
else:
    seed_predictions = np.load(saved_lightgbm)
    print(f'LightGBM-imputed model: loaded saved predictions of {len(seed_predictions)} seeds from {saved_lightgbm.name}')
lightgbm_imputed_prediction = seed_predictions.mean(axis=0)

LightGBM-imputed model: loaded saved predictions of 10 seeds from lightgbm_model_10_seeds.npy


## Part D: the CatBoost-imputed model

This is **the same model as Part C**, except the four models that fill in blanks are **CatBoost** instead of LightGBM (this was the original v5 recipe, which used seed 527). The two kinds of imputer guess missing values differently: their final predictions differ by about 0.45 per row. Averaging them cancels part of each one's mistakes. In cross-validation, the 50/50 average beat either one alone in every fold.

Everything else (practice copies with test blank patterns, test-mix weights, Ridge and gradient boosting) is inherited unchanged from Part C.

In [9]:
CATBOOST_CATEGORICAL = ['building_id', 'building_type', 'day_of_week']


def catboost_imputer_inputs(rows):
    """Same inputs as Part C, with the text columns kept as text (CatBoost's preferred format)."""
    inputs = imputer_inputs(rows)
    for column in CATBOOST_CATEGORICAL:
        inputs[column] = inputs[column].astype(str)
    return inputs


class CatBoostImputedModel(LightGBMImputedModel):
    def fit_imputers(self, training_rows, test_rows):
        all_inputs = pd.concat([training_rows.drop(columns='energy_usage'), test_rows], ignore_index=True)
        random_generator = np.random.default_rng(self.seed)
        weights = test_mix_weights(all_inputs, test_rows) ** self.test_mix_power
        weights = weights / weights.mean()
        test_blank_patterns = test_rows[NUMERIC_COLUMNS].isna().to_numpy()

        self.imputers = {}
        for column in NUMERIC_COLUMNS:
            observed = all_inputs.loc[all_inputs[column].notna()].copy()
            practice_sets = [observed]
            for _ in range(self.practice_copies):
                copy = observed.copy()
                blanks = test_blank_patterns[random_generator.integers(0, len(test_rows), len(copy))]
                blanks[:, NUMERIC_COLUMNS.index(column)] = False
                copy[NUMERIC_COLUMNS] = copy[NUMERIC_COLUMNS].mask(blanks)
                practice_sets.append(copy)
            practice_data = pd.concat(practice_sets, ignore_index=True)
            practice_weights = np.tile(weights[observed.index.to_numpy()], self.practice_copies + 1)

            imputer = catboost.CatBoostRegressor(iterations=1100, depth=5, learning_rate=0.05, loss_function='RMSE',
                                                 l2_leaf_reg=15.0, one_hot_max_size=20, random_seed=self.seed,
                                                 thread_count=4, verbose=False, allow_writing_files=False)
            imputer.fit(catboost_imputer_inputs(practice_data).drop(columns=column), practice_data[column],
                        sample_weight=practice_weights, cat_features=CATBOOST_CATEGORICAL)
            self.imputers[column] = imputer
        return self

    def expected_value(self, rows, column):
        return self.imputers[column].predict(catboost_imputer_inputs(rows).drop(columns=column))


saved_catboost = SAVED_FOLDER / 'catboost_imputed_test_predictions.npy'
if RETRAIN_COMPONENTS and catboost is not None:
    with threadpool_limits(limits=CPU_THREADS):
        catboost_imputed_prediction = CatBoostImputedModel(seed=527).fit(train, test_inputs).predict(test_inputs)
    print('CatBoost-imputed model: retrained')
else:
    catboost_imputed_prediction = np.load(saved_catboost)
    print(f'CatBoost-imputed model: loaded saved predictions from {saved_catboost.name}')

CatBoost-imputed model: loaded saved predictions from catboost_imputed_test_predictions.npy


## Part E: your friend's model

From your friend's code (`message.txt`), with **one change**: the 16 hand-set Ridge/CatBoost weights (one per blank pattern) are replaced by **one weight, 70% Ridge + 30% CatBoost**, chosen by cross-validation. The single weight scored as well as the 16, without their overfitting risk.

1. **CatBoost imputers** learn each numeric column from the other columns, using train and test inputs together (no practice copies, no weighting). Blanks in test rows are then filled.
2. **Ridge** (`alpha=3`) on a large set of features: per-building numeric slopes, a full hourly profile per building, day and month effects per building type, squares and pairwise products of the numeric columns, and numeric × hour effects. Trained on complete training rows.
3. **CatBoost** on the raw columns (building, type, hour, day, month and the four numeric values). Trained on complete training rows.

It's weaker than our model on its own, but its mistakes are different, so adding 25% of it improves the blend.

In [10]:
FRIEND_CATEGORICAL = ['building_id', 'building_type', 'day_of_week', 'hour', 'month']
FRIEND_THREADS = 8      # thread count used when the saved predictions were made


def friend_catboost_inputs(rows, numeric_columns):
    inputs = rows[['building_id', 'building_type', 'hour', 'day_of_week', 'month'] + numeric_columns].copy()
    for column in FRIEND_CATEGORICAL:
        inputs[column] = inputs[column].astype(str)
    return inputs


def friend_ridge_features(rows):
    """The friend's Ridge features (same columns, same order)."""
    index = rows.index
    def onehot(values, categories, prefix):
        return pd.get_dummies(pd.Categorical(values, categories=categories), prefix=prefix, dtype=float).set_index(index)
    building = onehot(rows['building_id'], BUILDING_IDS, 'building')
    building_type = onehot(rows['building_type'], BUILDING_TYPES, 'building_type')
    hour = onehot(rows['hour'], list(range(24)), 'hour')
    day = onehot(rows['day_of_week'], DAY_NAMES, 'day')
    month = onehot(rows['month'], list(range(1, 13)), 'month')

    interactions = {}
    for column in NUMERIC_COLUMNS:                       # per-building slopes
        for b in building.columns:
            interactions[f'{column}_x_{b}'] = rows[column].values * building[b].values
    for h in hour.columns:                               # a full hourly profile for every building
        for b in building.columns:
            interactions[f'{h}_x_{b}'] = hour[h].values * building[b].values
    for d in day.columns:                                # day of week per building type
        for t in building_type.columns:
            interactions[f'{d}_x_{t}'] = day[d].values * building_type[t].values
    for m in month.columns:                              # month per building type
        for t in building_type.columns:
            interactions[f'{m}_x_{t}'] = month[m].values * building_type[t].values
    for i, first in enumerate(NUMERIC_COLUMNS):          # squares and pairwise products
        interactions[f'{first}_squared'] = rows[first].values ** 2
        for second in NUMERIC_COLUMNS[i + 1:]:
            interactions[f'{first}_x_{second}'] = rows[first].values * rows[second].values
    for column in NUMERIC_COLUMNS:                       # numeric x hour
        for h in hour.columns:
            interactions[f'{column}_x_{h}'] = rows[column].values * hour[h].values
    return pd.concat([rows[NUMERIC_COLUMNS], building, building_type, hour, day, month,
                      pd.DataFrame(interactions, index=index)], axis=1)


def friend_model_predictions(training_rows, test_rows, ridge_share=0.7):
    # 1. CatBoost imputers on train + test inputs
    all_inputs = pd.concat([training_rows.drop(columns='energy_usage'), test_rows], ignore_index=True)
    imputers = {}
    for column in NUMERIC_COLUMNS:
        others = [c for c in NUMERIC_COLUMNS if c != column]
        observed = all_inputs[all_inputs[column].notna()]
        imputer = catboost.CatBoostRegressor(iterations=1800, depth=6, learning_rate=0.03, l2_leaf_reg=10, random_strength=0.4,
                                             loss_function='RMSE', random_seed=77, verbose=False,
                                             thread_count=FRIEND_THREADS, allow_writing_files=False)
        imputer.fit(friend_catboost_inputs(observed, others), observed[column], cat_features=FRIEND_CATEGORICAL)
        imputers[column] = imputer

    # 2 + 3. Ridge and CatBoost on complete training rows
    complete = training_rows[training_rows[NUMERIC_COLUMNS].notna().all(axis=1)].copy()
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=3)).fit(friend_ridge_features(complete), complete['energy_usage'])
    boosting = catboost.CatBoostRegressor(iterations=2200, depth=6, learning_rate=0.03, l2_leaf_reg=10, random_strength=0.5,
                                          loss_function='RMSE', random_seed=42, verbose=False,
                                          thread_count=FRIEND_THREADS, allow_writing_files=False)
    boosting.fit(friend_catboost_inputs(complete, NUMERIC_COLUMNS), complete['energy_usage'], cat_features=FRIEND_CATEGORICAL)

    # fill test blanks (two passes, as in the friend's code), then predict
    filled = test_rows.copy()
    for _ in range(2):
        for column in NUMERIC_COLUMNS:
            blank = filled[column].isna()
            if blank.sum():
                others = [c for c in NUMERIC_COLUMNS if c != column]
                filled.loc[blank, column] = imputers[column].predict(friend_catboost_inputs(filled.loc[blank], others))
    ridge_prediction = ridge.predict(friend_ridge_features(filled))
    boosting_prediction = boosting.predict(friend_catboost_inputs(filled, NUMERIC_COLUMNS))
    return ridge_share * ridge_prediction + (1 - ridge_share) * boosting_prediction


saved_friend = SAVED_FOLDER / 'friend_model_test_predictions.npz'
if RETRAIN_COMPONENTS and catboost is not None:
    friend_prediction = friend_model_predictions(train, test_inputs)
    print("friend's model: retrained")
else:
    friend_prediction = np.load(saved_friend)['friend_one_weight']
    print(f"friend's model: loaded saved predictions from {saved_friend.name}")

friend's model: loaded saved predictions from friend_model_test_predictions.npz


## Part F: the ensemble and the submission file

1. **Imputed models:** 50% LightGBM-imputed + 50% CatBoost-imputed.
2. **Our model:** 30% v4 + 70% imputed models.
3. **Final:** 75% our model + 25% friend's model.

| Recipe (cross-validation on train, test-style blanks) | Test-like blanks | Test-mix weighted |
|---|---|---|
| `candidate_final` (0.3 v4 + 0.7 LightGBM-imputed) | 3.2276 | 3.3705 |
| `candidate_combined` (+ 25% friend) | 3.2131 | 3.3613 |
| **`candidate_ensemble` (+ CatBoost-imputed half)** | **3.2117** | **3.3586** |

The last step improved the result in all 5 folds, but only by about 0.002 on average.

In [11]:
imputed_models = 0.5 * lightgbm_imputed_prediction + 0.5 * catboost_imputed_prediction
our_model = WEIGHT_V4_IN_OUR_MODEL * v4_prediction + (1 - WEIGHT_V4_IN_OUR_MODEL) * imputed_models
final_prediction = (1 - WEIGHT_FRIEND_MODEL) * our_model + WEIGHT_FRIEND_MODEL * friend_prediction
assert len(final_prediction) == len(test) and np.isfinite(final_prediction).all()
submission = pd.DataFrame({'id': test['id'], 'prediction': final_prediction})

if OUTPUT_FILE.exists():
    existing = pd.read_csv(OUTPUT_FILE)
    largest_difference = np.abs(existing['prediction'].to_numpy() - final_prediction).max()
    print(f'{OUTPUT_FILE.name} already exists, so it was not overwritten. Largest difference from this run: {largest_difference:.2e}')
else:
    submission.to_csv(OUTPUT_FILE, index=False)
    print(f'wrote {OUTPUT_FILE.name}')
print(f'{len(submission):,} rows | blanks: {submission["prediction"].isna().sum()}')
submission.head()

candidate_ensemble.csv already exists, so it was not overwritten. Largest difference from this run: 1.42e-14
3,000 rows | blanks: 0


,id,prediction
0,EC010541,71.748636
1,EC009755,66.237063
2,EC008886,55.992434
3,EC009317,64.335395
4,EC008958,21.354124


## What to expect

| File | Public RMSE |
|---|---|
| `best_model.csv` / `candidate_v8.csv` (one lucky seed) | 3.27026 |
| `candidate_final.csv` | 3.28 |
| `candidate_combined.csv` | about 3.28 |
| **`candidate_ensemble.csv`** | not submitted yet; expect about 3.27–3.28 |

**An honest note on 3.20:** every check we have, from cross-validation, the leaderboard and many experiments, puts this data's realistic floor around 3.26–3.28. The remaining error is mostly random noise, plus information that's genuinely missing where test values are blank. This ensemble is the most robust combination we've found, and nothing in it was tuned on the leaderboard, so its hidden-set score should match its public score closely. It can't promise 3.20.